# TD(λ)

TD(λ) は 1-step TD とモンテカルロの間を連続的につなぐ方法です。`λ` を動かすことで、短期更新と長期更新の混合比を滑らかに調整できます。


## 参考動画
- [強化学習プレイリスト（外部）](https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR)
@[youtube](https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR)

## 参考リンク
- https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR


## 背景と目的

`λ=0` は 1-step TD、`λ→1` は長い将来報酬を強く参照する更新になります。

タスクに応じて `λ` を調整できると、学習速度と安定性の両立がしやすくなります。

前向き視点の λ-return と後ろ向き視点の trace 更新を同じ軌跡で比較します。


In [ ]:
import math

gamma = 0.9
rewards = [0.4, -0.2, 0.8, 1.1]
v_boot = 0.5


def n_step_return(rews, n, gamma, v_boot):
    n = min(n, len(rews))
    g = 0.0
    for k in range(n):
        g += (gamma ** k) * rews[k]
    g += (gamma ** n) * v_boot
    return g


def lambda_return(rews, gamma, lam, v_boot):
    T = len(rews)
    out = 0.0
    for n in range(1, T + 1):
        w = (1 - lam) * (lam ** (n - 1)) if n < T else (lam ** (n - 1))
        out += w * n_step_return(rews, n, gamma, v_boot)
    return out

for lam in [0.0, 0.3, 0.7, 0.95]:
    print('lambda=', lam, 'G^lambda=', round(lambda_return(rewards, gamma, lam, v_boot), 6))


## 後ろ向き視点（Eligibility Trace）

\[e_t(s)=\gamma\lambda e_{t-1}(s)+\mathbf{1}[s_t=s],\quad V(s)\leftarrow V(s)+\alpha\,\delta_t\,e_t(s)\]

前向き計算を毎回やり直さず、オンラインで近い効果を得る実装です。


In [ ]:
alpha = 0.15
lam = 0.7
states = ['s0', 's1', 's0', 's2']
rewards = [0.3, 0.1, 0.7, 0.0]
V = {'s0': 0.2, 's1': 0.4, 's2': 0.1}
E = {'s0': 0.0, 's1': 0.0, 's2': 0.0}

for t in range(len(states) - 1):
    s = states[t]
    s_next = states[t + 1]
    r = rewards[t]
    delta = r + gamma * V[s_next] - V[s]

    for k in E:
        E[k] *= gamma * lam
    E[s] += 1.0

    for k in V:
        V[k] += alpha * delta * E[k]

print('Updated V =', {k: round(v, 6) for k, v in V.items()})
print('Final trace =', {k: round(v, 6) for k, v in E.items()})
